### Collect Tweets by ID

In [ ]:
from dotenv import load_dotenv
import requests
import json
import time
import os

load_dotenv()
auth_token = os.environ.get('AUTH_TOKEN')


url = "https://api.twitter.com/2/tweets/"


def get_tweets_by_id(id: str):
    querystring = {f"ids":{id},"tweet.fields":"author_id,created_at,geo,id,in_reply_to_user_id,lang,text","expansions":"author_id,geo.place_id,in_reply_to_user_id".format(id)}

    headers = {
        "Authorization": "Bearer " + auth_token
    }

    response = requests.request("GET", url, headers=headers, params=querystring)
   
    tweets = json.loads(response.content)


In [ ]:
import pandas as pd 

ids_trump_tweets = pd.read_csv('./data/ids.csv')

tweets = list()
for id in ids_trump_tweets.ids:
    response = (get_tweets_by_id(id))
    if 'data' in response:
       tweets.append(response['data'][0])

trump_tweets = pd.DataFrame(tweets)
trump_tweets

### Collect tweets by hashtags

In [1]:
from dotenv import load_dotenv
import urllib.request as urllib2
import pandas as pd
import requests
import json
import time
import os

load_dotenv()
auth_token = os.environ.get('AUTH_TOKEN')
header = {'Authorization': 'Bearer ' + auth_token}

class TwitterHook():

    def __init__(self, query, header = None, start_time = None, end_time = None, max_results= None):
        self.query = query
        self.header = header
        self.start_time = '2016-01-01T00%3A00%3A00Z'
        self.end_time = '2016-12-31T00%3A00%3A00Z'
        self.max_results = '500'

    def create_url(self):
        query = self.query
        start_time = self.start_time
        end_time = self.end_time        
        
        tweet_fields = "tweet.fields=author_id,id,created_at,in_reply_to_user_id,text"
        user_fields = "expansions=author_id&user.fields=id,name,username,created_at"
        start_time = (
            f"&start_time={self.start_time}"
            if self.start_time
            else ""
        )
        end_time = (
            f"&end_time={self.end_time}"
            if self.end_time
            else ""
        )
        max_results  = (
            f"&max_results={self.max_results}"
            if self.max_results
            else ""
        )
        url = "https://api.twitter.com/2/tweets/search/all?query={}&{}&{}{}{}{}".format(
               query, tweet_fields, user_fields, start_time, end_time, max_results
        )
        return url

    def connect_to_endpoint(self, url, header):
        response = requests.get(url,headers=header)
        listOfTweets = json.loads(response.content)
        return  listOfTweets


    def paginate(self, url, header, next_token=""):
        if next_token:
            full_url = f"{url}&next_token={next_token}"
            print('New Request on',full_url)
        else:
            full_url = url
            print('New Request on',full_url)
        data = self.connect_to_endpoint(full_url, header)
        yield data
        if "next_token" in data.get("meta", {}):
            yield from self.paginate(url, header, data['meta']['next_token'])


    def run(self):  
        url = self.create_url()
        yield from self.paginate(url, header)
        
        
def GetTweets(query):
    tweets = pd.DataFrame()
    for pg in TwitterHook(query).run():
        time.sleep(1)  
        
        if 'data' in pg:
            new_tweets = pd.DataFrame(pg['data'])
            tweets = pd.concat([tweets, new_tweets])
        else:
             print('Missing request')
    print('Done! Total of', len(tweets), 'tweets collected.')
    return tweets

In [2]:
#tweets = GetTweets(urllib2.quote('#trump2016 -rt'))
tweets = GetTweets(urllib2.quote('#DonaldTrump -rt'))

New Request on https://api.twitter.com/2/tweets/search/all?query=%23DonaldTrump%20-rt&tweet.fields=author_id,id,created_at,in_reply_to_user_id,text&expansions=author_id&user.fields=id,name,username,created_at&start_time=2016-01-01T00%3A00%3A00Z&end_time=2016-12-31T00%3A00%3A00Z&max_results=500
New Request on https://api.twitter.com/2/tweets/search/all?query=%23DonaldTrump%20-rt&tweet.fields=author_id,id,created_at,in_reply_to_user_id,text&expansions=author_id&user.fields=id,name,username,created_at&start_time=2016-01-01T00%3A00%3A00Z&end_time=2016-12-31T00%3A00%3A00Z&max_results=500&next_token=1jzu9lk96gu5npw3j120927k7vbv7kf7x6auqfryqsfx
New Request on https://api.twitter.com/2/tweets/search/all?query=%23DonaldTrump%20-rt&tweet.fields=author_id,id,created_at,in_reply_to_user_id,text&expansions=author_id&user.fields=id,name,username,created_at&start_time=2016-01-01T00%3A00%3A00Z&end_time=2016-12-31T00%3A00%3A00Z&max_results=500&next_token=1jzu9lk96gu5npw3j1206xs5txwn0qut8cnsqblgajr1
New

In [3]:
tweets.to_csv('donaldtrump.csv', index=False)

In [4]:
tweets

,id,text,created_at,edit_history_tweet_ids,author_id,in_reply_to_user_id,withheld
0,814984344204963841,Anonymous Just Declared War on Donald Trump Wi...,2016-12-30T23:59:43.000Z,[814984344204963841],2778043482,NaN,NaN
1,814984322885230592,Charles Koch Knocks Positions of Donald Trump ...,2016-12-30T23:59:38.000Z,[814984322885230592],2778043482,NaN,NaN
2,814984246989324288,Whose side is #donaldTrump on? #traitor Trump ...,2016-12-30T23:59:20.000Z,[814984246989324288],16124215,NaN,NaN
3,814983774794555392,Russia-US row: Trump Praises Putin Amid Hackin...,2016-12-30T23:57:28.000Z,[814983774794555392],782361008505556992,NaN,NaN
4,814983738518110208,@CNN is now selling a book about #DonaldTrump ...,2016-12-30T23:57:19.000Z,[814983738518110208],16362541,759251,NaN
...,...,...,...,...,...,...,...
479,808809801849438208,WASHINGTON - President-elect #DonaldTrump has ...,2016-12-13T23:04:18.000Z,[808809801849438208],3973558348,NaN,NaN
480,808809780588515336,@DianeRavitch short fiction on the impact of #...,2016-12-13T23:04:13.000Z,[808809780588515336],248324174,59010637,NaN
481,808809751060611072,Rick Perry is looking for the Energy Departmen...,2016-12-13T23:04:06.000Z,[808809751060611072],11720502,NaN,NaN
482,808809667375886336,No dejes de ver a @luisedavila analizando las ...,2016-12-13T23:03:46.000Z,[808809667375886336],2898618147,NaN,NaN
